# Data Preprocessing Pipeline - Road Accident Severity Prediction

This notebook outlines the entire data preparation pipeline to clean, encode, treat outliers, scale, and split our dataset.

## The Assembly Line:
1. **Time Extraction:** Pulling out numerical hours, days, and months.
2. **Encoding Categories:** Transforming word groups into numbers.
3. **Outlier Capping:** Squeezing wild weather sensor glitches down to safe limits.
4. **Train-Test Split:** Splitting the data into an 80% study group and a 20% secret test group.
5. **Feature Scaling:** Equalizing ranges to protect model mathematics.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("--- STEP 1: LOADING DATA ---")
# Open our baseline cleaned file
df = pd.read_csv("../data/processed/cleaned_accident_data.csv")

print("\n--- STEP 2: EXTRACTING TIME FEATURES ---")
# Convert raw text date strings into real time parameters
df['Start_Time'] = pd.to_datetime(df['Start_Time'])
df['Hour'] = df['Start_Time'].dt.hour
df['DayOfWeek'] = df['Start_Time'].dt.dayofweek
df['Month'] = df['Start_Time'].dt.month
df = df.drop(columns=['Start_Time'])

print(f"Matrix shape after time extraction: {df.shape}")

In [ ]:
print("--- STEP 3: ENCODING CATEGORICAL VARIABLES ---")
# A. Simple Category Indexing (for high-volume text columns to save memory)
df['Street']            = df['Street'].astype('category').cat.codes
df['City']              = df['City'].astype('category').cat.codes
df['Zipcode']           = df['Zipcode'].astype('category').cat.codes
df['Airport_Code']      = df['Airport_Code'].astype('category').cat.codes
df['Weather_Condition'] = df['Weather_Condition'].astype('category').cat.codes
df['Wind_Direction']    = df['Wind_Direction'].astype('category').cat.codes

# B. One-Hot Encoding (for small text columns using simple pd.get_dummies)
df = pd.get_dummies(df, columns=['Timezone', 'State'], dtype=int)

# C. Checkbox Conversion (turning True/False words into standard 1 and 0 integers)
df['Traffic_Signal'] = df['Traffic_Signal'].astype(int)
df['Junction']       = df['Junction'].astype(int)
df['Crossing']       = df['Crossing'].astype(int)
df['Stop']           = df['Stop'].astype(int)
df['Amenity']        = df['Amenity'].astype(int)
df['Bump']           = df['Bump'].astype(int)

print(f"Matrix shape after complete text encoding: {df.shape}")

In [ ]:
print("--- STEP 4: TREATING WEATHER OUTLIERS ---")
# Calculate the 99th percentile cutoff line to handle the broken 241.7 mph wind speed
wind_cutoff = df['Wind_Speed(mph)'].quantile(0.99)

# Clip everything outside our boundaries
df['Wind_Speed(mph)'] = np.clip(df['Wind_Speed(mph)'], 0, wind_cutoff)
print(f"Wind speed successfully capped at: {wind_cutoff} mph")

print("\n--- STEP 5: CREATING STRATIFIED SPLITS ---")
# Separate clues from answers
X = df.drop(columns=['Severity'])
y = df['Severity']

# 80% to train the model, 20% to test it fairly
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\n--- STEP 6: WEATHER FEATURE SCALING ---")
numeric_weather = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)']
scaler = StandardScaler()

# fit_transform learns rules strictly from X_train
X_train[numeric_weather] = scaler.fit_transform(X_train[numeric_weather])

# transform handles X_test using the same training rules (prevents data leakage!)
X_test[numeric_weather] = scaler.transform(X_test[numeric_weather])

print(f"\nFinal Preprocessed Training Set Dimensions: {X_train.shape}")
print(f"Final Preprocessed Testing Set Dimensions: {X_test.shape}")
print("\n--- PIPELINE EXECUTION SUCCESSFUL ---")